# Testing Tick Chart VS Time Chart

In [3]:
# Add the project root (one level up from /research) to the path
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
from utils.indicators import *
from utils.engine import *

# --- 1. Load Data ---
df_orderbook_1 = pd.read_csv("./csv/BTCUSDT_orderbook.csv")
df_trades_1 = pd.read_csv("./csv/BTCUSDT_trades.csv")

df_orderbook_2 = pd.read_csv("./csv/BTCUSDT_orderbook_2.csv")
df_trades_2 = pd.read_csv("./csv/BTCUSDT_trades_2.csv")

# Combine them vertically
df_orderbook = pd.concat([df_orderbook_1, df_orderbook_2], ignore_index=True)
df_trades = pd.concat([df_trades_1, df_trades_2], ignore_index=True)

# Convert unix timestamp to datetime
# Note: Keeping them as columns for merge_asof, we will set index later
df_orderbook['datetime'] = pd.to_datetime(df_orderbook['timestamp'], unit='ms')
df_trades['datetime'] = pd.to_datetime(df_trades['timestamp'], unit='ms')

# IMPORTANT: For pd.merge_asof, dataframes MUST be sorted by the timestamp
df_orderbook = df_orderbook.sort_values('datetime')
df_trades = df_trades.sort_values('datetime')

# Calculate trade directions before aggregation
df_trades['buy_vol']  = df_trades['amount'].where(df_trades['side'] == 'buy', 0)
df_trades['sell_vol'] = df_trades['amount'].where(df_trades['side'] == 'sell', 0)

# --- 2. Merge Orderbook into Trades ---
# This grabs the most recent Limit Order Book state for every single trade
df_merged = pd.merge_asof(
    df_trades, 
    df_orderbook[['datetime', 'bid_price', 'ask_price', 'bid_amount', 'ask_amount']], 
    on='datetime', 
    direction='backward'
)

# --- 3. Create Tick Groups ---
# Define how many trades make up one "Candle"
TICK_BAR_SIZE = 2000

# Create an arbitrary group ID for every N trades
df_merged['tick_group'] = np.arange(len(df_merged)) // TICK_BAR_SIZE

# --- 4. Aggregate into Tick Bars ---
df_tick_bars = df_merged.groupby('tick_group').agg({
    'datetime': 'last',             # Use the time of the last trade to timestamp the bar
    'price': ['first', 'max', 'min', 'last', 'count'],
    'amount': 'sum',
    'buy_vol': ['sum', 'max'],
    'sell_vol': ['sum', 'max'],
    'bid_price': 'last',            # State of the LOB at the close of the tick bar
    'ask_price': 'last',
    'bid_amount': 'last',
    'ask_amount': 'last'
})

# Flatten multi-level columns
# Flatten multi-level columns
df_tick_bars.columns = [
    'datetime', 'open', 'high', 'low', 'close', 'num_trades', 
    'volume', 'agg_buyer', 'largest_buyer', 
    'agg_seller', 'largest_seller', 
    'bid_price', 'ask_price', 'bid_amount', 'ask_amount'
]


# Now your index is 0, 1, 2, 3... and there are zero gaps between rows.
# Set datetime as the index so it behaves like a standard OHLCV dataframe
df_tick_bars = df_tick_bars.dropna(subset=['close']).reset_index(drop=True)

# --- 5. Feature Engineering (Now Tick-Based) ---
df_features = pd.DataFrame(index=df_tick_bars.index)

df_features['open'] = df_tick_bars['open']
df_features['high'] = df_tick_bars['high']
df_features['low'] = df_tick_bars['low']
df_features['close'] = df_tick_bars['close']

# Volume metrics
df_features['largest_buyer'] = df_tick_bars['largest_buyer'] * 100 / df_tick_bars['volume']
df_features['largest_seller'] = df_tick_bars['largest_seller'] * 100 / df_tick_bars['volume']

# Order Flow Imbalance
df_features['OFI_LOB'] = 100 * (df_tick_bars['bid_amount'] - df_tick_bars['ask_amount']) / (df_tick_bars['bid_amount'] + df_tick_bars['ask_amount'])
df_features['OFI_AGG'] = 100 * (df_tick_bars['agg_buyer'] - df_tick_bars['agg_seller']) / (df_tick_bars['agg_buyer'] + df_tick_bars['agg_seller'])

# Cumulative Volume Delta
df_features['CVD'] = calculate_cvd(df_tick_bars['agg_buyer'], df_tick_bars['agg_seller'])

# Candle Metrics
# Note: In tick bars, high and low can equal each other. Added a small epsilon (1e-9) to prevent division by zero errors.
candle_range = abs(df_tick_bars['high'] - df_tick_bars['low']) + 1e-9
df_features['candle_type'] = abs(df_tick_bars['close'] - df_tick_bars['open']) / candle_range
df_features['candle_size'] = candle_range / candle_range.rolling(14).mean()

# --- 6. Technical Indicators ---
price_clean = np.asanyarray(df_features['close'], dtype=float)
diffs = np.diff(price_clean)

# Calculate the raw RSI arrays
rsi_14_raw = get_wilders_rsi(diffs, 14)
rsi_14_sma_raw = sma(rsi_14_raw, 14)
rsi_2_raw = get_wilders_rsi(diffs, 2)

# Create full-length arrays filled with NaN
# Note: len(df_features) is the total number of rows.
# The RSI calculations produce (len - 1) values because they rely on np.diff().
rsi_14 = np.full(len(df_features), np.nan)
rsi_14_sma = np.full(len(df_features), np.nan)
rsi_2 = np.full(len(df_features), np.nan)

# Align arrays to the end
# Since np.diff() creates an array one shorter than the price array,
# we map them to the last (N-1) positions of our full-length array.
rsi_14[-(len(rsi_14_raw)):] = rsi_14_raw
rsi_14_sma[-(len(rsi_14_sma_raw)):] = rsi_14_sma_raw
rsi_2[-(len(rsi_2_raw)):] = rsi_2_raw # FIX: Use rsi_2_raw here, not rsi_2

# Assign to DataFrame
df_features['rsi_2'] = rsi_2
df_features['rsi_14'] = rsi_14
df_features['sma_14'] = rsi_14_sma

# Looking at Tick Chart

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure your data is the last 1000 ticks
df_plot = df_features.tail(1000)

# Create Subplots: Price (top), RSI 14 (mid), RSI 2 (bottom)
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.05, 
                    subplot_titles=('BTCUSDT Price', 'RSI 14', 'RSI 2'),
                    row_width=[0.2, 0.2, 0.6])

# --- Price Plot ---
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['close'], name='Price', line=dict(color='black', width=1)), row=1, col=1)

# --- RSI 14 Plot ---
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['rsi_14'], name='RSI 14', line=dict(color='blue', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['sma_14'], name='SMA 14', line=dict(color='orange', width=1)), row=2, col=1)

# --- RSI 2 Plot ---
fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot['rsi_2'], name='RSI 2', line=dict(color='purple', width=1)), row=3, col=1)

# Add Threshold lines
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)
fig.add_hline(y=90, line_dash="dash", line_color="red", row=3, col=1)
fig.add_hline(y=10, line_dash="dash", line_color="green", row=3, col=1)

# Update layout for "Trading View" feel
fig.update_layout(
    height=800,
    template="plotly_white",
    hovermode="x unified", # Shows all values on hover
    xaxis_rangeslider_visible=False # Set to True if you want a slider at the bottom
)

fig.show()